### Attribute-Based Access Control (Control de Acceso Basado en Atributos)
Este cuaderno muestra cómo implementar la seguridad a nivel de datos mediante el Control de Acceso Basado en Atributos (ABAC).

En los ejemplos anteriores, los filtros de fila y las máscaras de columna se configuraron directamente en tablas individuales. Aunque este método es eficaz, requiere definir y administrar las políticas de seguridad por separado para cada tabla.

ABAC permite abordar esta limitación mediante un modelo centralizado en el que:

- Los datos se clasifican mediante etiquetas gobernadas que representan distintos atributos.
- Las políticas de seguridad se definen una sola vez en función de esos atributos.
- Las políticas se aplican automáticamente a múltiples objetos de datos, según las condiciones y el ámbito configurados.

Los objetivos de esta implementación son:

- Clasificar la información confidencial mediante etiquetas de gobernanza.
- Definir políticas de seguridad reutilizables basadas en dichas etiquetas.

#### Paso 1: Usar el schema "data_protection" en el catalog "demo"

In [0]:
USE CATALOG demo;
USE SCHEMA data_protection;

#### Paso 2: Crear la tabla "customers_privacy"
Se crea una tabla que almacena información de clientes procedentes de distintos países.

In [0]:
CREATE OR REPLACE TABLE customers_abac (
    customer_id INT,
    name STRING,
    country STRING,
    email STRING,
    balance DECIMAL(10,2),
    credit_card STRING
);

#### Paso 3: Insertar datos de muestra
El conjunto de datos contiene registros correspondientes a los países de Perú y Estados Unidos.

Esto permite comprobar el funcionamiento de la seguridad a nivel de fila, verificando que cada usuario solo pueda visualizar los datos correspondientes a su región.

In [0]:
INSERT INTO customers_abac VALUES
(2001, 'Carlos', 'Perú', 'carlos@email.com', 8500.00, '4532123456789012'),
(2002, 'Ana', 'Perú', 'ana@email.com', 6200.00, '5214789632147852'),
(2003, 'John', 'USA', 'john@email.com', 9300.00, '6011122233334444'),
(2004, 'Isabel', 'USA', 'isabel@email.com', 9900.00, '6011122233335555'),
(2005, 'Luci', 'Mexico', 'maria@email.com',  4100.00, '3782822463100050');

In [0]:
SELECT * FROM customers_abac;

#### Paso 4: Crear un UDF para Row Filter y Column Mask
(Estas funciones ya se crearon durante los capítulos anteiores).

1. `fn_filter_country`: Evalúa la pertenencia del usuario a un grupo y determina si un registro específico debe incluirse en los resultados según el país correspondiente.
2. `fn_mask_email`: Comprueba a qué grupo pertenece el usuario y determina si debe devolver el valor original del correo electrónico o una versión enmascarada.
3. `fn_mask_credit_card`: Comprueba a qué grupo pertenece el usuario y determina si debe devolver el valor original de la tarjeta de crédito o una versión enmascarada.

#### Paso 5: Crear etiquetas gobernadas
- `pii = email`, `pii = credit_card`: Indica que los datos contienen información de **identificación personal (PII)**, en este caso, direcciones de correo electrónico, la tarjeta de crédito, etc.

- `access_type = country`: Especifica el tipo de control de acceso que se aplicará a los datos, como el filtrado basado en el país, departamento, etc.

#### Paso 6: Crear Política
(Estas políticas pueden aplicarse a nivel de catálogo, esquema o tabla).

- `policy_filter_country`: Política encargada de filtrar los registros según el país correspondiente.
- `policy_mask_email`: Política responsable de enmascarar las direcciones de correo electrónico según las reglas de acceso establecidas.
- `policy_mask_credit_card`: Política responsable de enmascarar las tarjetas de crédito según las reglas de acceso establecidas.

#### Paso 7: Asociar una Etiqueta a la Tabla
- customers_abac.country
- customers_abac.email
- customers_abac.credit_card

In [0]:
SELECT * FROM customers_abac;

#### Paso 8: Crear una tabla adicional para demostrar como ABAC puede escalarse a múltiples objetos de datos

In [0]:
CREATE OR REPLACE TABLE customers_abac_scalability (
    customer_id INT,
    name STRING,
    country STRING,
    email STRING,
    balance DECIMAL(10,2),
    credit_card STRING
);

In [0]:
INSERT INTO customers_abac_scalability VALUES
(2001, 'Carlos', 'Perú', 'carlos@email.com', 8500.00, '4532123456789012'),
(2002, 'Ana', 'Perú', 'ana@email.com', 6200.00, '5214789632147852'),
(2003, 'John', 'USA', 'john@email.com', 9300.00, '6011122233334444'),
(2004, 'Isabel', 'USA', 'isabel@email.com', 9900.00, '6011122233335555'),
(2005, 'Luci', 'Mexico', 'maria@email.com',  4100.00, '3782822463100050');

In [0]:
ALTER TABLE customers_abac_scalability
ALTER COLUMN email
SET TAGS ('pii' = 'email');

In [0]:
ALTER TABLE customers_abac_scalability
ALTER COLUMN credit_card
SET TAGS ('pii' = 'credit_card');

In [0]:
ALTER TABLE customers_abac_scalability
ALTER COLUMN country
SET TAGS ('access_type' = 'country');

In [0]:
SELECT * FROM customers_abac_scalability;